# Week 19, Day 1: Conjugate priors: beta-binomial and gamma-Poisson

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w019/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Posterior for a release probability

With a Beta(2, 2) prior and 7 releases in 20 trials, compute `post_a`, `post_b`, the 95% credible interval by sampling 200,000 values with `rng.beta` (`ci_samples`) and on a fine grid (`ci_grid`). Then update one trial at a time through the sequence in `trials` and store the final parameters in `seq_a` and `seq_b`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2007)
a0, b0, k, n = 2, 2, 7, 20
post_a, post_b = ..., ...
ci_samples = np.percentile(rng.beta(post_a, post_b, 200000), [2.5, 97.5])

p = np.linspace(0.0005, 0.9995, 20000)
dens = p ** (post_a - 1) * (1 - p) ** (post_b - 1)
cdf = np.cumsum(dens) / dens.sum()
ci_grid = ...                                  # the grid points where the CDF crosses 0.025 and 0.975

trials = np.array([1] * 7 + [0] * 13); rng.shuffle(trials)
seq_a, seq_b = a0, b0
for t in trials:
    ...
print(f"posterior Beta({post_a}, {post_b}), mean {post_a / (post_a + post_b):.3f};  95% interval {ci_samples.round(3)} (grid {np.round(ci_grid, 3)});  sequential Beta({seq_a}, {seq_b})")

In [ ]:
import numpy as np
def test_params():
    "post_a and post_b follow the conjugate update"
    assert post_a == 2 + 7 and post_b == 2 + 13

def test_interval():
    "the sampled 95% credible interval agrees with the grid interval to within 0.01"
    assert np.allclose(ci_samples, ci_grid, atol=0.01)

def test_sequential():
    "updating one trial at a time gives the same posterior as updating with all the data at once"
    assert (seq_a, seq_b) == (post_a, post_b)

test_params()
test_interval()
test_sequential()
print("All checks passed")

### Exercise: Bayes beats the plain average

100 neurons have rates drawn from Gamma(shape 4, rate 0.8) (mean 5 spikes per trial). Each is recorded on just 2 trials. Compute each neuron's plain average `mle`, its posterior mean `post_mean` under that gamma prior, and the mean squared errors of both against the true rates.

In [ ]:
a_pop, b_pop = 4.0, 0.8
true_rates = rng.gamma(a_pop, 1 / b_pop, 100)          # numpy uses shape and scale = 1 / rate
counts = rng.poisson(true_rates[:, None], (100, 2))
mle = ...
post_mean = ...
mse_mle = ...
mse_bayes = ...
print(f"mean squared error: per-neuron average {mse_mle:.2f}, posterior mean {mse_bayes:.2f}")
plt.plot(true_rates, mle, ".", label="average of 2 trials"); plt.plot(true_rates, post_mean, ".", label="posterior mean")
plt.plot([0, 15], [0, 15], "k--"); plt.xlabel("true rate"); plt.ylabel("estimate"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_posterior():
    "post_mean is (a + total count) / (b + n) for every neuron"
    assert np.allclose(post_mean, (a_pop + counts.sum(axis=1)) / (b_pop + 2))

def test_better():
    "the posterior means have a lower mean squared error than the per-neuron averages"
    assert mse_bayes < 0.8 * mse_mle

def test_shrink():
    "posterior means are pulled towards the population mean: their spread is smaller than the averages' spread"
    assert post_mean.std() < mle.std()

test_posterior()
test_better()
test_shrink()
print("All checks passed")